# 🔥 PINN — Inverse Heat Source Identification (1D)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/pinn-inverse-heat/blob/main/notebooks/01_inverse_1D_colab.ipynb)

**Problem**: recover unknown source $f(x)$ from sparse noisy measurements of $T(x)$:
$$-\frac{d^2T}{dx^2} = f(x), \quad x\in(0,1), \quad T(0)=T(1)=0$$

In [ ]:
# ═════════════════════════════════════════════════════════
#  CELL 1 — Colab Setup (run this first)         
# ═════════════════════════════════════════════════════════
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Clone repo to get src/ modules
    if not os.path.exists('/content/pinn-inverse-heat'):
        !git clone https://github.com/MaximeAuger/pinn-inverse-heat.git
    sys.path.insert(0, '/content/pinn-inverse-heat/src')
    
    # Create results dir in Colab's writable space
    RESULTS_DIR = '/content/results'
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f'Running on Colab. Results → {RESULTS_DIR}')
else:
    sys.path.insert(0, '../src')
    RESULTS_DIR = '../results'
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f'Running locally. Results → {RESULTS_DIR}')

In [ ]:
# ══════════════════════════════════════════════════════
#  CELL 2 — Imports
# ══════════════════════════════════════════════════════
import gc
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

torch.manual_seed(42)
np.random.seed(42)

# ── Memory helpers ──────────────────────────────────
def free_memory():
    """Force garbage collection — call after each major block."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def mem_mb():
    """Rough RSS memory usage in MB (Linux/Colab)."""
    try:
        import resource
        return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024
    except Exception:
        return -1

print(f'PyTorch {torch.__version__} | RAM usage: {mem_mb():.0f} MB')

In [ ]:
# ══════════════════════════════════════════════════════
#  CELL 3 — Architecture
# ══════════════════════════════════════════════════════

class MLP(nn.Module):
    """Generic fully-connected network with Tanh activations."""
    def __init__(self, layers, init_scale=1.0):
        super().__init__()
        net = []
        for i in range(len(layers) - 1):
            linear = nn.Linear(layers[i], layers[i+1])
            # Scaled Xavier init — init_scale=0.1 keeps SourceNet near zero
            nn.init.xavier_normal_(linear.weight)
            linear.weight.data *= init_scale
            nn.init.zeros_(linear.bias)
            net.append(linear)
            if i < len(layers) - 2:
                net.append(nn.Tanh())
        self.net = nn.Sequential(*net)

    def forward(self, x):
        return self.net(x)


def make_models():
    pinn   = MLP([1, 64, 64, 64, 1], init_scale=1.0)
    source = MLP([1, 64, 64, 64, 1], init_scale=0.1)  # starts near f≈0
    return pinn, source

pinn, source_net = make_models()
n = sum(p.numel() for p in pinn.parameters())
print(f'Parameters per network: {n:,}')

In [ ]:
# ══════════════════════════════════════════════════════
#  CELL 4 — Ground truth & observations
# ══════════════════════════════════════════════════════

def f_true(x):
    return np.sin(np.pi * x) + 0.5 * np.sin(3 * np.pi * x)

def T_true(x):
    return (1/np.pi**2)*np.sin(np.pi*x) + (0.5/(9*np.pi**2))*np.sin(3*np.pi*x)

N_OBS  = 20
NOISE  = 0.01

x_obs_np = np.linspace(0.05, 0.95, N_OBS)
T_clean  = T_true(x_obs_np)
T_obs_np = T_clean + NOISE * np.max(np.abs(T_clean)) * np.random.randn(N_OBS)

# All tensors created ONCE — reused throughout, no duplication
x_obs    = torch.tensor(x_obs_np, dtype=torch.float32).unsqueeze(1)
T_obs    = torch.tensor(T_obs_np, dtype=torch.float32).unsqueeze(1)
x_colloc = torch.linspace(0, 1, 200).unsqueeze(1)   # 200 pts (was 300)
x_eval   = torch.linspace(0, 1, 300).unsqueeze(1)   # eval grid
x_plot   = np.linspace(0, 1, 300)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_plot, T_true(x_plot), 'b-', lw=2)
axes[0].scatter(x_obs_np, T_obs_np, c='red', s=50, zorder=5, label=f'{N_OBS} obs')
axes[0].set_title('T*(x) — partially observed'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(x_plot, f_true(x_plot), 'r-', lw=2)
axes[1].set_title('f*(x) — to recover (unknown)'); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/ground_truth.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
free_memory()

In [ ]:
# ══════════════════════════════════════════════════════
#  CELL 5 — Loss function (memory-safe)
# ══════════════════════════════════════════════════════

def compute_losses(pinn, source_net, x_colloc, x_obs, T_obs,
                   w_pde=1.0, w_bc=500.0, w_data=1000.0, w_reg=1e-4):

    # PDE residual  -T'' - f = 0
    # IMPORTANT: clone + requires_grad only on local var, not stored globally
    x_c = x_colloc.detach().clone().requires_grad_(True)
    T   = pinn(x_c)
    dT  = torch.autograd.grad(T,  x_c, torch.ones_like(T),  create_graph=True)[0]
    d2T = torch.autograd.grad(dT, x_c, torch.ones_like(dT), create_graph=True)[0]
    f_c = source_net(x_colloc)
    L_pde = torch.mean((d2T + f_c) ** 2)

    # Boundary conditions
    x0 = torch.zeros(1, 1); x1 = torch.ones(1, 1)
    L_bc = pinn(x0).squeeze() ** 2 + pinn(x1).squeeze() ** 2

    # Data fidelity
    L_data = torch.mean((pinn(x_obs) - T_obs) ** 2)

    # Tikhonov order-1  ||df/dx||²
    x_r = x_colloc.detach().clone().requires_grad_(True)
    f_r = source_net(x_r)
    df  = torch.autograd.grad(f_r, x_r, torch.ones_like(f_r), create_graph=True)[0]
    L_reg = torch.mean(df ** 2)

    total = w_pde * L_pde + w_bc * L_bc + w_data * L_data + w_reg * L_reg

    return {
        'total': total,                  # tensor  — used for .backward()
        'pde':   L_pde.item(),           # float   — logged, NO graph retained
        'bc':    L_bc.item(),
        'data':  L_data.item(),
        'reg':   L_reg.item(),
    }

print('Loss function defined ✓')

In [ ]:
# ══════════════════════════════════════════════════════
#  CELL 6 — Training  (Adam → L-BFGS)
#
#  Memory fixes applied
# ══════════════════════════════════════════════════════

torch.manual_seed(42)
pinn, source_net = make_models()

all_params = list(pinn.parameters()) + list(source_net.parameters())
optimizer  = optim.Adam(all_params, lr=5e-4)

ADAM_EPOCHS = 6000
SNAP_EVERY  = 300

# ── Python lists of FLOATS only — zero tensor memory ──
history   = {k: [] for k in ['total', 'pde', 'bc', 'data', 'reg']}
snapshots = []   # list of (label, x_np, T_np, f_np) — all numpy

print('Phase 1 — Adam')
print(f'{"Epoch":>7} | {"Total":>9} | {"PDE":>9} | {"Data":>9} | RAM')
print('─' * 55)

for epoch in range(1, ADAM_EPOCHS + 1):

    # Curriculum: ramp PDE weight over first 1500 epochs
    w_pde = min(1.0, epoch / 1500.0)

    optimizer.zero_grad()
    L = compute_losses(pinn, source_net, x_colloc, x_obs, T_obs,
                       w_pde=w_pde, w_bc=500.0, w_data=1000.0, w_reg=1e-4)
    L['total'].backward()   # frees the graph immediately
    torch.nn.utils.clip_grad_norm_(all_params, max_norm=1.0)
    optimizer.step()

    # Log as floats — NOT tensors
    for k in history:
        val = L[k].item() if hasattr(L[k], 'item') else L[k]
        history[k].append(val)

    # Snapshot: convert to numpy immediately, release tensors
    if epoch % SNAP_EVERY == 0 or epoch == 1:
        with torch.no_grad():
            T_s = pinn(x_eval).squeeze().numpy().copy()
            f_s = source_net(x_eval).squeeze().numpy().copy()
        snapshots.append((epoch, x_eval.squeeze().numpy().copy(), T_s, f_s))

    # Periodic GC
    if epoch % 1000 == 0:
        free_memory()
        print(f'{epoch:>7} | {L["total"]:>9.3e} | {L["pde"]:>9.3e} | '
              f'{L["data"]:>9.3e} | {mem_mb():.0f} MB')

print(f'\nAdam done. RAM: {mem_mb():.0f} MB')
free_memory()

In [ ]:
# ══════════════════════════════════════════════════════
#  CELL 7 — L-BFGS fine-tuning
# ══════════════════════════════════════════════════════

print('Phase 2 — L-BFGS')
opt_lbfgs = optim.LBFGS(all_params, lr=0.5, max_iter=50,
                        history_size=50, line_search_fn='strong_wolfe')
lbfgs_log = []

def closure():
    opt_lbfgs.zero_grad()
    L = compute_losses(pinn, source_net, x_colloc, x_obs, T_obs,
                       w_pde=1.0, w_bc=500.0, w_data=1000.0, w_reg=1e-4)
    L['total'].backward()
    lbfgs_log.append(L['total'].item())   # float only
    return L['total']

for step in range(300):
    opt_lbfgs.step(closure)
    if (step + 1) % 100 == 0:
        free_memory()
        print(f'  step {step+1:3d} | loss {lbfgs_log[-1]:.4e} | RAM {mem_mb():.0f} MB')

# Final snapshot
with torch.no_grad():
    T_final = pinn(x_eval).squeeze().numpy().copy()
    f_final = source_net(x_eval).squeeze().numpy().copy()
snapshots.append(('final', x_eval.squeeze().numpy().copy(), T_final, f_final))

free_memory()
print(f'\nTraining complete ✓  |  RAM: {mem_mb():.0f} MB')

In [ ]:
# ══════════════════════════════════════════════════════
#  CELL 8 — Final results
# ══════════════════════════════════════════════════════
x_np = x_eval.squeeze().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(x_plot, T_true(x_plot), 'b-', lw=2.5, label='Analytical $T^*$')
ax.plot(x_np, T_final, 'r--', lw=2, label='PINN')
ax.scatter(x_obs_np, T_obs_np, c='gray', s=40, zorder=5, alpha=0.8, label='Obs.')
ax.set_title('Temperature $T(x)$', fontsize=13); ax.set_xlabel('x')
ax.legend(); ax.grid(alpha=0.3)
T_err = np.abs(T_final - T_true(x_np))
ax.text(0.05, 0.95, f'Max: {T_err.max():.2e}\nL²: {np.mean(T_err**2)**0.5:.2e}',
        transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8), fontsize=10)

ax = axes[1]
ax.plot(x_plot, f_true(x_plot), 'b-', lw=2.5, label='True $f^*$')
ax.plot(x_np, f_final, 'r--', lw=2, label='PINN recovered')
ax.set_title('Source $f(x)$ — recovered', fontsize=13); ax.set_xlabel('x')
ax.legend(); ax.grid(alpha=0.3)
f_err = np.abs(f_final - f_true(x_np))
ax.text(0.05, 0.95, f'Max: {f_err.max():.2e}\nL²: {np.mean(f_err**2)**0.5:.2e}',
        transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8), fontsize=10)

plt.suptitle('PINN Inverse Problem — Final Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/final_results.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
free_memory()

In [ ]:
# ══════════════════════════════════════════════════════
#  CELL 9 — Loss history
# ══════════════════════════════════════════════════════
epochs = np.arange(1, ADAM_EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].semilogy(epochs, history['total'], 'k-', lw=1.5)
axes[0].axvline(1500, color='orange', ls='--', alpha=0.7, label='PDE ramp-up ends')
axes[0].set_title('Total loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)

for k, label in [('pde','PDE'),('bc','BC'),('data','Data'),('reg','Tikhonov')]:
    axes[1].semilogy(epochs, history[k], label=label)
axes[1].set_title('Loss decomposition'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/loss_history.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
free_memory()

In [ ]:
# ══════════════════════════════════════════════════════
#  CELL 10 — Training animation (GIF)
#  Note: rendered at low dpi to save RAM
# ══════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

def animate(i):
    lbl, xs, Ts, fs = snapshots[i]
    for ax in axes: ax.cla()
    axes[0].plot(x_plot, T_true(x_plot), 'b-', lw=2, alpha=0.5, label='$T^*$')
    axes[0].plot(xs, Ts, 'r-', lw=2, label='PINN')
    axes[0].scatter(x_obs_np, T_obs_np, c='gray', s=25, zorder=5)
    axes[0].set_ylim(-0.005, 0.14)
    axes[0].set_title(f'T(x) — epoch {lbl}'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(x_plot, f_true(x_plot), 'b-', lw=2, alpha=0.5, label='$f^*$')
    axes[1].plot(xs, fs, 'r-', lw=2, label='PINN')
    axes[1].set_ylim(-1.8, 1.8)
    axes[1].set_title(f'f(x) — epoch {lbl}'); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()

ani = animation.FuncAnimation(fig, animate, frames=len(snapshots), interval=200)
gif_path = f'{RESULTS_DIR}/training_animation.gif'
ani.save(gif_path, writer='pillow', fps=5, dpi=80)   # low dpi = less RAM
plt.close()
free_memory()
print(f'GIF saved ✓  ({os.path.getsize(gif_path)//1024} KB)')
from IPython.display import Image
Image(gif_path)

In [ ]:
# ══════════════════════════════════════════════════════
#  CELL 11 — Noise sensitivity
#  Memory-safe: one model at a time, deleted after use
# ══════════════════════════════════════════════════════

def quick_train(x_obs_t, T_obs_t, epochs=3000, w_reg=1e-4):
    """Train fresh models and return only the final f prediction (numpy)."""
    torch.manual_seed(0)
    _pinn, _src = make_models()
    _params = list(_pinn.parameters()) + list(_src.parameters())
    _opt = optim.Adam(_params, lr=5e-4)

    for ep in range(1, epochs + 1):
        ramp = min(1.0, ep / 1000.0)
        _opt.zero_grad()
        L = compute_losses(_pinn, _src, x_colloc, x_obs_t, T_obs_t,
                           w_pde=ramp, w_bc=500.0, w_data=1000.0, w_reg=w_reg)
        L['total'].backward()
        torch.nn.utils.clip_grad_norm_(_params, 1.0)
        _opt.step()
        if ep % 1000 == 0:
            free_memory()

    with torch.no_grad():
        f_pred = _src(x_eval).squeeze().numpy().copy()

    # Explicitly delete models to free RAM before next run
    del _pinn, _src, _params, _opt
    free_memory()
    return f_pred


noise_levels  = [0.0, 0.01, 0.02, 0.05, 0.10]
results_noise = {}

for noise in noise_levels:
    np.random.seed(0)
    T_n   = T_clean + noise * np.max(np.abs(T_clean)) * np.random.randn(N_OBS)
    T_n_t = torch.tensor(T_n, dtype=torch.float32).unsqueeze(1)
    f_pred = quick_train(x_obs, T_n_t)
    l2 = float(np.mean((f_pred - f_true(x_np))**2)**0.5)
    results_noise[noise] = (f_pred, l2)
    print(f'  noise={noise*100:5.1f}%  →  f L² = {l2:.4f}  | RAM {mem_mb():.0f} MB')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cmap = plt.cm.plasma
axes[0].plot(x_plot, f_true(x_plot), 'k-', lw=3, label='$f^*$', zorder=10)
for i, (noise, (fp, _)) in enumerate(results_noise.items()):
    axes[0].plot(x_np, fp, '--', lw=1.5, color=cmap(i/len(noise_levels)),
                 label=f'{noise*100:.0f}%')
axes[0].set_title('Source recovery vs noise'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot([n*100 for n in noise_levels],
             [results_noise[n][1] for n in noise_levels], 'ro-', ms=8)
axes[1].set_xlabel('Noise (%)'); axes[1].set_ylabel('L² error'); axes[1].grid(alpha=0.3)
axes[1].set_title('L² error vs noise level')
plt.suptitle('Noise Sensitivity Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/noise_sensitivity.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
free_memory()

In [ ]:
# ══════════════════════════════════════════════════════
#  CELL 12 — Tikhonov regularization study
# ══════════════════════════════════════════════════════

reg_weights = [0.0, 1e-5, 1e-4, 1e-2]
results_reg = {}

np.random.seed(0)
T_noisy   = T_clean + 0.03 * np.max(np.abs(T_clean)) * np.random.randn(N_OBS)
T_noisy_t = torch.tensor(T_noisy, dtype=torch.float32).unsqueeze(1)

for wr in reg_weights:
    f_pred = quick_train(x_obs, T_noisy_t, epochs=3000, w_reg=wr)
    l2 = float(np.mean((f_pred - f_true(x_np))**2)**0.5)
    results_reg[wr] = (f_pred, l2)
    print(f'  w_reg={wr:.0e}  →  f L² = {l2:.4f}  | RAM {mem_mb():.0f} MB')

plt.figure(figsize=(9, 5))
plt.plot(x_plot, f_true(x_plot), 'k-', lw=3, label='$f^*$ true')
cmap2 = plt.cm.viridis
for i, (wr, (fp, l2)) in enumerate(results_reg.items()):
    plt.plot(x_np, fp, '--', lw=1.8, color=cmap2(i/len(reg_weights)),
             label=f'$w_{{reg}}={wr:.0e}$ (L²={l2:.3f})')
plt.title('Effect of Tikhonov Regularization (noise=3%)', fontsize=12)
plt.xlabel('x'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/tikhonov_study.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
free_memory()

In [ ]:
# ══════════════════════════════════════════════════════
#  CELL 13 — Summary & optional Drive export
# ══════════════════════════════════════════════════════
import glob

print('═' * 45)
print('  RESULTS SUMMARY')
print('═' * 45)
T_err_final = np.abs(T_final - T_true(x_np))
f_err_final = np.abs(f_final - f_true(x_np))
print(f'  Temperature  L² error : {np.mean(T_err_final**2)**0.5:.2e}')
print(f'  Source       L² error : {np.mean(f_err_final**2)**0.5:.2e}')
print(f'  Observations           : {N_OBS} pts, noise={NOISE*100:.0f}%')
print(f'  Adam epochs            : {ADAM_EPOCHS}')
print(f'  L-BFGS steps           : 300')
print(f'  Peak RAM (approx)      : {mem_mb():.0f} MB')
print('═' * 45)
print('Generated files:')
for fp in sorted(glob.glob(f'{RESULTS_DIR}/*')):
    print(f'  {os.path.basename(fp):40s} {os.path.getsize(fp)//1024:>5} KB')

if IN_COLAB:
    print('\nTo save to Google Drive:')
    print('  from google.colab import drive')
    print('  drive.mount("/content/drive")')
    print('  !cp -r /content/results /content/drive/MyDrive/')